setup

Defines the volume location and a shortcut that loads a bronze table.

In [0]:
from pyspark.sql import functions as F

catalog = "workspace"
schema = "de_ecommerce"
base = "/Volumes/workspace/de_ecommerce/raw_files"
holding = f"{base}/holding"

def bronze(name):
    return spark.table(f"{catalog}.{schema}.bronze_{name}")

assign each order to a batch

Takes each order and labels it initial, batch_1, batch_2, or batch_3 based on its purchase date. It then counts the orders per batch so you can see the sizes.

In [0]:
batches = (bronze("orders").select("order_id", "order_purchase_timestamp")
    .withColumn("batch",
        F.when(F.col("order_purchase_timestamp") < "2018-01-01", "initial")
         .when(F.col("order_purchase_timestamp") < "2018-04-01", "batch_1")
         .when(F.col("order_purchase_timestamp") < "2018-07-01", "batch_2")
         .otherwise("batch_3"))
    .select("order_id", "batch"))

display(batches.groupBy("batch").count().orderBy("batch"))

batch,count
batch_1,21208
batch_2,19979
batch_3,12824
initial,45430


write each chunk to the volume

For each of the 4 tables and each of the 4 batches, it takes the rows belonging to that batch (by joining on order_id), drops the tracking columns, and saves them as one Parquet file in holding/<table>/<batch>/.

In [0]:
meta = ["source_file", "ingested_at"]
tables = ["orders", "order_items", "order_payments", "order_reviews"]
batch_names = ["initial", "batch_1", "batch_2", "batch_3"]

for t in tables:
    for b in batch_names:
        (bronze(t).drop(*meta)
            .join(batches.filter(F.col("batch") == b), "order_id")
            .drop("batch")
            .coalesce(1)
            .write.mode("overwrite")
            .parquet(f"{holding}/{t}/{b}"))
    print("written:", t)

written: orders
written: order_items
written: order_payments
written: order_reviews


check the totals

Reads each saved chunk back and counts its rows. For each table, the batch counts should add up to the total in the bronze table.

In [0]:
for t in tables:
    counts = {b: spark.read.parquet(f"{holding}/{t}/{b}").count() for b in batch_names}
    print(t, counts, "| total:", sum(counts.values()), "| bronze:", bronze(t).count())

orders {'initial': 45430, 'batch_1': 21208, 'batch_2': 19979, 'batch_3': 12824} | total: 99441 | bronze: 99441
order_items {'initial': 51234, 'batch_1': 24097, 'batch_2': 22978, 'batch_3': 14341} | total: 112650 | bronze: 112650
order_payments {'initial': 47871, 'batch_1': 22027, 'batch_2': 20763, 'batch_3': 13225} | total: 103886 | bronze: 103886
order_reviews {'initial': 45370, 'batch_1': 21190, 'batch_2': 19889, 'batch_3': 12775} | total: 99224 | bronze: 99224
